In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.transformer_lstm_svm import Transformer_LSTM_SVM

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_NAME = "Transformer_LSTM_SVM"


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-3, weight_decay=1e-4,
                 seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    # two-stage model
    model = Transformer_LSTM_SVM(n_classes=8, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({MODEL_NAME}) ===")
        print(f"  shapes: train={tuple(X_tr.shape)} val={tuple(X_va.shape)} "
              f"test={tuple(X_te.shape)}")
        print(f"  train labels: {np.bincount(y_tr.numpy())}")
        print(f"  backbone params: {n_params:,}")
        print("  --- Stage 1: training Transformer-LSTM backbone ---")

    # stage 1: train transformer-LSTM with softmax head
    with Timer(device) as stage1_timer:
        model.fit_backbone(X_tr, y_tr, X_val=X_va, y_val=y_va,
                           epochs=epochs, lr=lr, weight_decay=weight_decay,
                           batch_size=50, seed=seed, verbose=verbose)

    if verbose:
        print("  --- Stage 2: fitting RBF-SVM on FC(8) logits ---")

    # stage 2: fit SVM on backbone's FC(8) logits (CPU-bound)
    with Timer(device=None) as stage2_timer:
        model.fit_svm(X_tr, y_tr)

    peak_mem_mb = get_gpu_peak_memory_mb(device)
    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed

    # inference timing
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # test evaluation
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(backbone)={stage1_timer.elapsed:.1f}s  "
              f"stage2(SVM)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario":  scenario_idx,
        "model":     MODEL_NAME,
        "n_train":   len(X_tr),
        "accuracy":  acc,
        "precision": p,
        "recall":    r,
        "f1":        f,
        "confusion": cm,
        "n_params":  n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }



In [3]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-3, seed=0)
    results.append(r)


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



=== Scenario 1 (Transformer_LSTM_SVM) ===
  shapes: train=(7858, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [ 858 1000 1000 1000 1000 1000 1000 1000]
  backbone params: 5,339
  --- Stage 1: training Transformer-LSTM backbone ---
  [stage1] ep   1 | loss 2.0651 acc 0.147 | val loss 1.9767 acc 0.177
  [stage1] ep  10 | loss 1.7431 acc 0.313 | val loss 1.5496 acc 0.371
  [stage1] ep  20 | loss 1.2139 acc 0.502 | val loss 1.1798 acc 0.519
  [stage1] ep  30 | loss 0.4276 acc 0.816 | val loss 0.4148 acc 0.849
  [stage1] ep  40 | loss 0.3391 acc 0.848 | val loss 0.3537 acc 0.858
  [stage1] ep  50 | loss 0.3204 acc 0.859 | val loss 0.3403 acc 0.866
  [stage1] ep  60 | loss 0.3065 acc 0.865 | val loss 0.3284 acc 0.871
  [stage1] ep  70 | loss 0.2975 acc 0.866 | val loss 0.3230 acc 0.871
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.8675  P=0.8703  R=0.8675  F1=0.8672
  COMPUTE: stage1(backbone)=111.5s  stage2(SVM)=0.3s  total=111.8s  inf=0.121ms/sample

=== 

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  [stage1] ep   1 | loss 2.0799 acc 0.139 | val loss 2.0710 acc 0.172
  [stage1] ep  10 | loss 1.9182 acc 0.207 | val loss 1.9212 acc 0.236
  [stage1] ep  20 | loss 1.4978 acc 0.403 | val loss 1.4195 acc 0.425
  [stage1] ep  30 | loss 1.2940 acc 0.477 | val loss 1.3199 acc 0.474
  [stage1] ep  40 | loss 1.2523 acc 0.493 | val loss 1.2721 acc 0.496
  [stage1] ep  50 | loss 1.2204 acc 0.497 | val loss 1.2429 acc 0.495
  [stage1] ep  60 | loss 1.2135 acc 0.501 | val loss 1.2356 acc 0.504
  [stage1] ep  70 | loss 1.2038 acc 0.519 | val loss 1.2280 acc 0.502
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.5131  P=0.5707  R=0.5131  F1=0.5011
  COMPUTE: stage1(backbone)=55.1s  stage2(SVM)=0.5s  total=55.5s  inf=0.138ms/sample

=== Scenario 3 (Transformer_LSTM_SVM) ===
  shapes: train=(786, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [ 86 100 100 100 100 100 100 100]
  backbone params: 5,339
  --- Stage 1: training Transformer-LSTM backbone ---
  [stage1] ep   

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


...

  [stage1] ep  10 | loss 2.0745 acc 0.166 | val loss 2.0719 acc 0.172
  [stage1] ep  20 | loss 2.0266 acc 0.156 | val loss 1.9814 acc 0.182
  [stage1] ep  30 | loss 1.9752 acc 0.161 | val loss 1.9650 acc 0.181
  [stage1] ep  40 | loss 1.9701 acc 0.179 | val loss 1.9613 acc 0.186
  [stage1] ep  50 | loss 1.9696 acc 0.174 | val loss 1.9603 acc 0.190
  [stage1] ep  60 | loss 1.9687 acc 0.205 | val loss 1.9598 acc 0.190
  [stage1] ep  70 | loss 1.9658 acc 0.205 | val loss 1.9597 acc 0.197
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.2162  P=0.2715  R=0.2163  F1=0.1630
  COMPUTE: stage1(backbone)=4.5s  stage2(SVM)=0.0s  total=4.5s  inf=0.021ms/sample

=== Scenario 5 (Transformer_LSTM_SVM) ===
  shapes: train=(238, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [28 30 30 30 30 30 30 30]
  backbone params: 5,339
  --- Stage 1: training Transformer-LSTM backbone ---
  [stage1] ep   1 | loss 2.0866 acc 0.118 | val loss 2.0820 acc 0.126


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  [stage1] ep  10 | loss 2.0833 acc 0.147 | val loss 2.0770 acc 0.148
  [stage1] ep  20 | loss 2.0700 acc 0.151 | val loss 2.0699 acc 0.173
  [stage1] ep  30 | loss 2.0711 acc 0.160 | val loss 2.0620 acc 0.177
  [stage1] ep  40 | loss 2.0610 acc 0.151 | val loss 2.0420 acc 0.179
  [stage1] ep  50 | loss 2.0395 acc 0.143 | val loss 2.0188 acc 0.179
  [stage1] ep  60 | loss 2.0147 acc 0.185 | val loss 2.0005 acc 0.177
  [stage1] ep  70 | loss 2.0279 acc 0.189 | val loss 1.9905 acc 0.176
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.1975  P=0.1578  R=0.1975  F1=0.1491
  COMPUTE: stage1(backbone)=3.5s  stage2(SVM)=0.0s  total=3.5s  inf=0.013ms/sample

=== Scenario 6 (Transformer_LSTM_SVM) ===
  shapes: train=(77, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [ 7 10 10 10 10 10 10 10]
  backbone params: 5,339
  --- Stage 1: training Transformer-LSTM backbone ---
  [stage1] ep   1 | loss 2.0870 acc 0.104 | val loss 2.0860 acc 0.124


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  [stage1] ep  10 | loss 2.0815 acc 0.117 | val loss 2.0792 acc 0.134
  [stage1] ep  20 | loss 2.0763 acc 0.182 | val loss 2.0802 acc 0.135
  [stage1] ep  30 | loss 2.0731 acc 0.143 | val loss 2.0813 acc 0.127
  [stage1] ep  40 | loss 2.0560 acc 0.182 | val loss 2.0832 acc 0.126
  [stage1] ep  50 | loss 2.0662 acc 0.182 | val loss 2.0832 acc 0.130
  [stage1] ep  60 | loss 2.0728 acc 0.143 | val loss 2.0833 acc 0.128
  [stage1] ep  70 | loss 2.0687 acc 0.169 | val loss 2.0833 acc 0.128
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.1706  P=0.2718  R=0.1706  F1=0.1389
  COMPUTE: stage1(backbone)=1.5s  stage2(SVM)=0.0s  total=1.5s  inf=0.008ms/sample


In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "train_sec_stage1",
                       "train_sec_stage2", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios ===")
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)



=== Transformer_LSTM_SVM summary across scenarios ===
 scenario                model  n_train  accuracy  precision  recall     f1  n_params  train_sec  train_sec_stage1  train_sec_stage2  inf_ms_per_sample  peak_mem_mb
        1 Transformer_LSTM_SVM     7858    0.8675     0.8703  0.8675 0.8672      5339     111.80            111.50              0.29             0.1208        314.4
        2 Transformer_LSTM_SVM     3939    0.5131     0.5707  0.5131 0.5011      5339      55.55             55.05              0.50             0.1378        169.6
        3 Transformer_LSTM_SVM      786    0.2331     0.3191  0.2331 0.2007      5339      11.64             11.59              0.05             0.0388         84.3
        4 Transformer_LSTM_SVM      391    0.2162     0.2715  0.2163 0.1630      5339       4.51              4.50              0.01             0.0210         84.3
        5 Transformer_LSTM_SVM      238    0.1975     0.1578  0.1975 0.1491      5339       3.49              3.48      

In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])


Scenario 1  (Transformer_LSTM_SVM, n_train=7858, acc=0.868)
[[136   0   0  63   0   0   1   0]
 [  1 183   0   1   0   1   0  14]
 [  0   0 194   0   0   6   0   0]
 [ 27   0   0 171   0   0   2   0]
 [  0   0   0   0 200   0   0   0]
 [  0   2   3   0   0 154   0  41]
 [  0   0   0   1   1   0 198   0]
 [  0  14   0   0   0  34   0 152]]

Scenario 2  (Transformer_LSTM_SVM, n_train=3939, acc=0.513)
[[  0  11  37  94   3   0   5  50]
 [  0  77  53   9   0   0   0  61]
 [  0   0 155   0   0   0   0  45]
 [  0   9  37 106   1   0   7  40]
 [  0   0   0   0 200   0   0   0]
 [  0   3  57   1   0  97   0  42]
 [  0   6  14  58   1   0  94  27]
 [  0   2 105   1   0   0   0  92]]

Scenario 3  (Transformer_LSTM_SVM, n_train=786, acc=0.233)
[[  0   0 100   0   0   1  99   0]
 [  0  79  77   0   0   3  41   0]
 [  0   0 114   0   6   1  79   0]
 [  0   2 100   0   1   1  96   0]
 [  0   0  42   0  15   1 142   0]
 [  0  28  57   0   1  67  47   0]
 [  0   0 101   0   1   0  98   0]
 [  0   1 1